<a href="https://colab.research.google.com/github/mostofa89/TrafficCongestionRiskZones./blob/main/TrafficCongestionRiskZones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import os
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

#Loading Datasets

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ucimachinelearning/vanet-traffic-congestion-dataset")

print("Path to dataset files:", path)

In [ ]:
print(os.listdir(path))

In [ ]:
df = pd.read_csv(os.path.join(path, "vanet_traffic_data.csv"))

print("DataFrame created successfully!")
print("Shape:", df.shape)

display(df.head())

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.shape

#Data PreProcessing

In [ ]:
df.columns

In [ ]:
df.isnull().sum()

In [ ]:
df.columns.tolist()

In [ ]:
df.isnull().sum()

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

In [ ]:
df.head()

In [ ]:
df.dtypes

##Feature Engineering

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df.head()

In [ ]:
df["timestamp_hour"] = df["timestamp"].dt.hour
df["timestamp_day_of_week"] = df["timestamp"].dt.dayofweek
df["timestamp_month"] = df["timestamp"].dt.month

df.head()

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

df["timestamp_hour"] = df["timestamp"].dt.hour
df["timestamp_day_of_week"] = df["timestamp"].dt.dayofweek

df = df.drop("timestamp", axis=1)

In [ ]:
df.head()

In [ ]:
print(df["timestamp_month"].value_counts())

In [ ]:
df = df.drop("timestamp_month", axis=1)

In [ ]:
print("Road segments:")
print(df["road_segment_id"].unique())

print("\nLabels:")
print(df["label"].unique())

In [ ]:
df = df.drop("road_segment_id", axis=1)

In [ ]:
df.head()

In [ ]:
le = LabelEncoder()

df['label_encoded'] = le.fit_transform(df['label'])
print("Label Mapping:")

for number, label in enumerate(le.classes_):
  print(f"{number} = {label}")

In [ ]:
df.duplicated().sum()

## Graphical Analysis

In [ ]:
sns.countplot(data=df, x="label")
plt.title("Traffic Congestion Class Distribution")
plt.xlabel("Congestion Level")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.stripplot(
    data=df,
    x="label",
    y="avg_speed_kmph",
    jitter=True,
    alpha=0.5
)

plt.title("Average Speed vs Traffic Congestion Level")
plt.xlabel("Congestion Level")
plt.ylabel("Average Speed (km/h)")
plt.show()

In [ ]:
sns.stripplot(
    data=df,
    x="label",
    y="avg_speed_kmph",
    jitter=0.25,
    alpha=0.15,
    size=2
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x="density_veh_per_km",
    y="avg_speed_kmph",
    hue="label",
    alpha=0.5
)

plt.title("Traffic Density vs Average Speed")
plt.xlabel("Vehicle Density (vehicles/km)")
plt.ylabel("Average Speed (km/h)")
plt.show()

## Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include="number").drop(columns=["label_encoded"], errors="ignore")
corr = numeric_df.corr()

In [ ]:
plt.figure(figsize=(18, 14))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    linewidths=0.5
)

plt.title("Correlation Heatmap of Traffic Features")
plt.show()

##Outlier Analysis

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns

numeric_cols = numeric_cols.drop(
    "label_encoded",
    errors="ignore"
)

for col in numeric_cols:

    plt.figure(figsize=(8, 4))

    sns.boxplot(x=df[col])

    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)

    plt.tight_layout()
    plt.show()

In [ ]:
for col in numeric_cols:
  Q1 = df[col].quantile(0.25)
  Q3 = df[col].quantile(0.75)

  IQR = Q3 - Q1
  lower_bound = Q3 - IQR * 1.5
  upper_bound = Q3 + IQR * 1.5
  print(f"\n{col}")
  print("Q1:", Q1)
  print("Q3:", Q3)
  print("IQR:", IQR)
  print("Lower Bound:", lower_bound)
  print("Upper Bound:", upper_bound)
  outliers = df[
    (df[col] < lower_bound) |
    (df[col] > upper_bound)
  ]

  print("Number of outliers:", len(outliers))


In [ ]:
features = [
    "avg_wait_time_s",
    "flow_veh_per_hr",
    "queue_length_veh",
    "avg_accel_ms2",
    "temp_c",
    "rssi_dbm",
    "acceleration_directionality",
    "congestion_pressure"
]

for col in features:
    plt.figure(figsize=(16, 4))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot - {col}")
    plt.show()

## Outlier Detection using IQR

In [ ]:
outlier_results = []

for col in numeric_cols:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outlier_count = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    ).sum()

    outlier_percentage = (
        outlier_count / len(df) * 100
    )

    outlier_results.append([
        col,
        Q1,
        Q3,
        IQR,
        lower_bound,
        upper_bound,
        outlier_count,
        outlier_percentage
    ])

outlier_df = pd.DataFrame(
    outlier_results,
    columns=[
        "Feature",
        "Q1",
        "Q3",
        "IQR",
        "Lower Bound",
        "Upper Bound",
        "Outlier Count",
        "Outlier %"
    ]
)

display(outlier_df)

## Skewness Analysis

In [ ]:
for col in features:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution - {col}")
    plt.show()

In [ ]:
skewness = (
  df[numeric_cols].skew().sort_values(ascending=False)
)

display(skewness.to_frame('skewness'))

##Invalid Value Check

In [ ]:
for col in ["avg_wait_time_s", "queue_length_veh", "congestion_pressure"]:
  print(
    col,
    "minimum =", df[col].min(),
    "negative values =", (df[col] < 0).sum()
  )

##Feature Selection

In [ ]:
df[col] = np.log1p(df[col])

In [ ]:
ml_df = df.copy()
X = ml_df.drop(
    columns = ["label", "label_encoded"],
    errors = "ignore",

)
X = X.select_dtypes(include = ["number"])

y = ml_df["label_encoded"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size = .2,
    random_state = 42,
    stratify = y
)

## Appling Nomalization

In [ ]:
robust_scaler = RobustScaler(
    quantile_range = (0.25, 0.75)
)

X_train_scaled = robust_scaler.fit_transform(X_train)
X_test_scaled = robust_scaler.fit_transform(X_test)

In [ ]:
print("Original training shape:", X_train.shape)
print("Scaled training shape:", X_train_scaled.shape)
print("Scaled testing shape:", X_test_scaled.shape)

print("\nFirst 5 scaled rows:")
print(X_train_scaled[:5])

#Model Training and Testing

In [ ]:
k_range = range(2, 10)

# Store inertia (Within-Cluster Sum of Squares)
inertia = []

for k in k_range:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    km.fit(X_train_scaled)
    inertia.append(km.inertia_)

In [ ]:
# Plot Elbow Curve
plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o')

plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for Optimal K")
plt.xticks(k_range)
plt.grid(True)

plt.show()

In [ ]:
best_k = 4

kmeans = KMeans(
    n_clusters = best_k,
    random_state = 42,
    n_init = 10
)

cluster_labels = kmeans.fit_predict(X_train_scaled)
print(f"Selected K is {best_k}")

In [ ]:
cluster_df = X_train.copy()

cluster_df["Cluster"] = cluster_labels

cluster_df.head(10)

In [ ]:
# Convert cluster centers back to original scale
cluster_centers = robust_scaler.inverse_transform(kmeans.cluster_centers_)

centers_df = pd.DataFrame(
    cluster_centers,
    columns=X_train.columns
)

In [ ]:
plt.figure(figsize=(12, 7))

sns.scatterplot(
    data=cluster_df,
    x="density_veh_per_km",
    y="avg_speed_kmph",
    hue="Cluster",
    palette="viridis",
    s=60,
    alpha=0.55,
    edgecolor="white",
    linewidth=0.3
)

sns.scatterplot(
    data=centers_df,
    x="density_veh_per_km",
    y="avg_speed_kmph",
    color="red",
    marker="*",
    s=300,
    edgecolor="black",
    linewidth=1.5,
    label="Cluster Center"
)

plt.title(
    "K-Means Clustering of Traffic Conditions",
    fontsize=18,
    fontweight="bold"
)

plt.xlabel("Vehicle Density (vehicles/km)", fontsize=13)
plt.ylabel("Average Speed (km/h)", fontsize=13)

plt.legend(
    title="Cluster",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

plt.grid(
    True,
    linestyle="--",
    alpha=0.25
)

plt.tight_layout()
plt.show()